In [ ]:
!pip install ultralytics==8.2.0 -q

import os, torch, shutil
from pathlib import Path
import torch.serialization
import ultralytics
from ultralytics import YOLO

try:
    torch.serialization.add_safe_globals([ultralytics.nn.tasks.DetectionModel])
except Exception:
    pass

_orig_torch_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_torch_load(*args, **kwargs)
torch.load = _safe_load

print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

model = YOLO('yolov8n.pt')
print('Model loaded successfully! Starting VisDrone training on Kaggle GPU...')
results = model.train(
    data='VisDrone.yaml',
    epochs=5,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='/tmp/sutra_train',
    name='visdrone_yolov8n',
    exist_ok=True,
    save=True,
    verbose=True
)
print('Training completed!')

best_pt = Path('/tmp/sutra_train/visdrone_yolov8n/weights/best.pt')
if best_pt.exists():
    best_m = YOLO(str(best_pt))
    best_m.export(format='onnx', imgsz=640)
    shutil.copy(str(best_pt), '/kaggle/working/best.pt')
    onnx_file = Path('/tmp/sutra_train/visdrone_yolov8n/weights/best.onnx')
    if onnx_file.exists():
        shutil.copy(str(onnx_file), '/kaggle/working/best.onnx')
    print('SUCCESS: best.pt and best.onnx ready in /kaggle/working!')
